In [1]:
import random
random.seed(42)

# Modell 11 LSTM with Keras

### Was kann dieses Modell?

### Daten: alle Rennen
### Modelle: LSTM
### Training: sequentiell
### Vorhersage: Probs für alle Fahrer
### Problem: ???

## Imports

In [2]:
import fastf1
from fastf1 import plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# sklearn related imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss
from sklearn.utils.class_weight import compute_sample_weight



project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from utils.Feature_Engineer import FeatureEngineer
from utils.Metrics import Metrics
from utils.Models import Models
from utils.experiment_db import init_db
init_db()


pd.set_option('display.width', None)         # passt die Ausgabe-Breite an
pd.set_option('display.max_columns', None)   # zeigt alle Spalten
pd.set_option('display.max_rows', None)      # zeigt alle Reihen

%load_ext autoreload
%autoreload 2

## Rennen definieren

In [3]:
schedule = fastf1.get_event_schedule(2024, include_testing=False)
print(schedule["EventName"].tolist())

req         WARNING 	DEFAULT CACHE ENABLED! (2.5 GB) /Users/lukaskoch/Library/Caches/fastf1


['Bahrain Grand Prix', 'Saudi Arabian Grand Prix', 'Australian Grand Prix', 'Japanese Grand Prix', 'Chinese Grand Prix', 'Miami Grand Prix', 'Emilia Romagna Grand Prix', 'Monaco Grand Prix', 'Canadian Grand Prix', 'Spanish Grand Prix', 'Austrian Grand Prix', 'British Grand Prix', 'Hungarian Grand Prix', 'Belgian Grand Prix', 'Dutch Grand Prix', 'Italian Grand Prix', 'Azerbaijan Grand Prix', 'Singapore Grand Prix', 'United States Grand Prix', 'Mexico City Grand Prix', 'São Paulo Grand Prix', 'Las Vegas Grand Prix', 'Qatar Grand Prix', 'Abu Dhabi Grand Prix']


In [4]:
YEAR = 2024
#GRAND_PRIX = ['Austrian Grand Prix', 'British Grand Prix', 'Hungarian Grand Prix', 'Belgian Grand Prix']
GRAND_PRIX = schedule['EventName'].to_list()
SESSION_TYPE = "R"

In [5]:
all_laps = []
for gp in GRAND_PRIX:
    try:
        session = fastf1.get_session(YEAR, gp, SESSION_TYPE)
        session.load(weather=True, laps=True, telemetry=True)
        laps = session.laps.copy()
        weather = session.weather_data.copy()
        
        laps = laps.sort_values("Time")
        weather = weather.sort_values("Time")

        merged = pd.merge_asof(
                laps,
                weather,
                on="Time",
                direction="backward"
            )
        
        merged['GP'] = gp
        merged['Year'] = YEAR
        
        all_laps.append(merged)
        print(f"✔ Loaded {gp}")
        
    except Exception as e:
        print(f"Error loading {gp}: {e}")
        
data = pd.concat(all_laps, ignore_index=True)

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']
core           INFO 	Loading data for Saudi Arabian Grand Prix

✔ Loaded Bahrain Grand Prix


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '16', '81', '14', '63', '38', '4', '44', '27', '23', '20', '31', '2', '22', '3', '77', '24', '18', '10']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


✔ Loaded Saudi Arabian Grand Prix


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 19 drivers: ['55', '16', '4', '81', '11', '18', '22', '14', '27', '20', '23', '3', '10', '77', '24', '31', '63', '44', '1']
core           INFO 	Loading data for Japanese Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


✔ Loaded Australian Grand Prix


... (38 outputs left out)

req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '81', '63', '10', '55', '14', '24', '20', '4', '77', '44', '22', '30', '23', '27', '11', '18', '43', '31']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


✔ Loaded Qatar Grand Prix


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '55', '16', '44', '63', '1', '10', '27', '14', '81', '23', '22', '24', '18', '61', '20', '30', '77', '43', '11']


✔ Loaded Abu Dhabi Grand Prix


In [6]:
fe = FeatureEngineer()
merged = fe.create_features(data)

In [7]:
features = fe.choose_features(merged)
features

['gapAheadSeconds',
 's1_delta',
 's2_delta',
 's3_delta',
 'lap_pace_delta',
 'is_raining',
 'position_change',
 'compound_HARD',
 'compound_INTERMEDIATE',
 'compound_MEDIUM',
 'compound_SOFT',
 'compound_WET',
 'status_1',
 'status_12',
 'status_124',
 'status_125',
 'status_1254',
 'status_126',
 'status_1264',
 'status_1267',
 'status_14',
 'status_16',
 'status_167',
 'status_21',
 'status_24',
 'status_26',
 'status_267',
 'status_2671',
 'status_4',
 'status_41',
 'status_412',
 'status_45',
 'status_51',
 'status_6',
 'status_64',
 'status_671',
 'status_71']

In [8]:
race_order = (
    merged.groupby(["GP", "Year"])["LapStartDate"]
      .min()
      .sort_values()
      .reset_index()[["GP", "Year"]]
)

race_order

,GP,Year
0,Bahrain Grand Prix,2024
1,Saudi Arabian Grand Prix,2024
2,Australian Grand Prix,2024
3,Japanese Grand Prix,2024
4,Chinese Grand Prix,2024
5,Miami Grand Prix,2024
6,Emilia Romagna Grand Prix,2024
7,Monaco Grand Prix,2024
8,Canadian Grand Prix,2024
9,Spanish Grand Prix,2024


In [9]:
train_races = race_order.iloc[:3]
test_race = race_order.iloc[-1:]

train_df = merged.merge(train_races, on=["GP", "Year"]).sort_values(["GP", "Year", "LapNumber", "Driver"])
test_df = merged.merge(test_race, on=["GP", "Year"]).sort_values(["GP", "Year", "LapNumber", "Driver"])

train_df[features] = train_df[features].astype('float32')
test_df[features]  = test_df[features].astype('float32')

In [10]:
X_train_raw = train_df[features].values
y_train_raw = train_df['pit_in_next_lap'].values

X_test_raw = test_df[features].values
y_test_raw = test_df['pit_in_next_lap'].values

In [11]:
SEQ_LEN = 10

def create_sequences(features, labels, seq_len):
    X = []
    y = []
    for i in range(len(features) - seq_len):
        X.append(features[i:i + seq_len])
        y.append(labels[i + seq_len])
    return np.array(X), np.array(y) 


X_train, y_train = create_sequences(X_train_raw, y_train_raw, SEQ_LEN)
X_test, y_test = create_sequences(X_test_raw, y_test_raw, SEQ_LEN)

In [12]:
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import LSTM, Dense, Masking # type: ignore
from tensorflow.keras.optimizers import Adam # type: ignore
from tensorflow.keras.metrics import AUC, Precision, Recall # type: ignore

In [13]:
model = Sequential([
    LSTM(64, input_shape=(SEQ_LEN, len(features))),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(1e-3),
    loss='binary_crossentropy',
    metrics=[
        AUC(name="auc"), 
        Precision(name="precision"), 
        Recall(name="recall")]
)


model.summary()

/Users/lukaskoch/anaconda3/envs/base_3_12/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        26,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,177 (102.25 KB)

 Trainable params: 26,177 (102.25 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(zip(np.unique(y_train), class_weights))
print(class_weights)

{0: 0.5167808219178082, 1: 15.39795918367347}


In [15]:
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    class_weight= class_weights
)

Epoch 1/100
85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.6498 - loss: 0.6754 - precision: 0.0632 - recall: 0.5532 - val_auc: 0.4560 - val_loss: 0.5215 - val_precision: 0.0233 - val_recall: 0.2500
Epoch 2/100
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.7402 - loss: 0.6155 - precision: 0.0682 - recall: 0.6489 - val_auc: 0.3142 - val_loss: 0.6564 - val_precision: 0.0141 - val_recall: 0.2500
Epoch 3/100
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.7810 - loss: 0.5689 - precision: 0.0949 - recall: 0.6702 - val_auc: 0.2852 - val_loss: 0.5171 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 4/100
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.8086 - loss: 0.5400 - precision: 0.1040 - recall: 0.6915 - val_auc: 0.2789 - val_loss: 0.7071 - val_precision: 0.0093 - val_recall: 0.2500
Epoch 5/100
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.8129 - loss: 0.5389 - precision: 0.0914 - recall: 0.7128 - val_auc: 0.2756 - val_loss: 0.4666 - val_precision: 0.0000e+00 - val_recall: 

In [16]:
y_pred_prob = model.predict(X_test)

33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [17]:
from utils.experiment_db import log_or_update_metrics

metrics = Metrics.calculate_metrics(f"approach_11_LSTM", y_test, y_pred_prob)
Metrics.print_metrics(metrics)
    
    
log_or_update_metrics(
    model_name=f"approach_11_{LSTM}",
    metrics={
        **metrics
    }
)

   ✔ Brier-Score:              0.0471 -> optimal 0.0
   ✔ Log-Loss:                 0.3052 -> optimal 0.0
   ✔ ROC-AUC:                  0.5362 -> optimal 1.0
   ✔ PR-AUC:                   0.0455 -> optimal 1.0
